# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [3]:
# 创建三个参数，都从 0 开始，启用梯度
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b  = torch.tensor(0.0, requires_grad=True)

# SGD 优化器，管理这三个参数
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

for epoch in range(1, 2001):
    # 1) 设置本轮学习率 = 0.1 / sqrt(epoch)，越往后步子越小
    for g in optimizer.param_groups:
        g['lr'] = 0.1 / (epoch ** 0.5)

    # 2) 清空上一轮残留的梯度
    optimizer.zero_grad()

    # 3) 前向：调用 predict，同时传入 x1、x2
    y_pred = predict(x1, x2, w1, w2, b)

    # 4) 损失：逐样本绝对误差，再用 .mean() 求平均
    per_sample_loss = torch.abs(y - y_pred)
    loss = per_sample_loss.mean()

    # 5) 反向传播，自动算出每个参数的梯度
    loss.backward()

    # 6) 优化器更新 w1、w2、b
    optimizer.step()

# 训练完，在 no_grad() 里用最终参数重新算一遍（这次不算梯度）
with torch.no_grad():
    y_pred_final = predict(x1, x2, w1, w2, b)
    per_sample_loss_final = torch.abs(y - y_pred_final)
    loss_final = per_sample_loss_final.mean()

# 打印所有结果和 shape
print('w1 =', w1.item(), ' shape =', tuple(w1.shape))
print('w2 =', w2.item(), ' shape =', tuple(w2.shape))
print('b  =', b.item(),  ' shape =', tuple(b.shape))
print('x1 shape =', tuple(x1.shape))
print('x2 shape =', tuple(x2.shape))
print('y  shape =', tuple(y.shape))
print('y_pred =', y_pred_final)
print('y_pred shape =', tuple(y_pred_final.shape))
print('逐样本损失 =', per_sample_loss_final)
print('逐样本损失 shape =', tuple(per_sample_loss_final.shape))
print('平均损失 =', loss_final.item())
print('平均损失 shape =', tuple(loss_final.shape))


w1 = 1.999006986618042  shape = ()
w2 = -3.0010085105895996  shape = ()
b  = 0.9993674159049988  shape = ()
x1 shape = (6,)
x2 shape = (6,)
y  shape = (6,)
y_pred = tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954])
y_pred shape = (6,)
逐样本损失 = tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046])
逐样本损失 shape = (6,)
平均损失 = 0.0023727715015411377
平均损失 shape = ()


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。
1.2个，x1,x2的长度是6，代表一共有6个样本
2.多加了w2*x2,参数多了w2:损失算法不变，优化器步骤也没变
3.损失不断变小，预测值靠近真实值，就说明训练有效
4.参数初始值为0的时候也可以算出梯度